In [1]:
import os
from itertools import cycle
import google.generativeai as genai
import time
from google.api_core.exceptions import ResourceExhausted
from dotenv import load_dotenv

load_dotenv()

# List of API keys
api_keys = [
    os.getenv("GENAI_API_KEY_5"),
    os.getenv("GENAI_API_KEY_4"),
    os.getenv("GENAI_API_KEY_3"),
    os.getenv("GENAI_API_KEY_2"),
    os.getenv("GENAI_API_KEY_1"),
]
api_keys_cycle = cycle(api_keys)

# Check if all API keys are set
for i, key in enumerate(api_keys):
    if not key:
        raise ValueError(f"API key {i+1} not found. Please set the GENAI_API_KEY_{i+1} environment variable.")

def generate_image_prompt(text):
    while True:
        try:
            api_key = next(api_keys_cycle)
            # print(f"Using API key: {api_key}")
            genai.configure(api_key=api_key)
            system_instruction = (
                """You are an image prompt generating AI assistant. You are asked to generate a prompt for stable diffusion image generation based on the given dialogue that explains the scenario. 
                It should be not long, short and concise, and should contain only the most important elements of the scene. Only use at most 2 people.
                You should describe the characters' physically like what they are wearing, their expressions and their actions and not use their names. 
                An example output is: 'masterpiece, best quality, 1girl, collarbone, wavy hair, looking at viewer, blurry foreground, upper body, necklace, contemporary, plain pants, intricate, print, pattern, ponytail, freckles, red hair, dappled sunlight, smile, happy,'"""
            )
            model = genai.GenerativeModel(model_name="gemini-1.5-flash-8b", system_instruction=system_instruction)
            generation_config = genai.GenerationConfig(
                max_output_tokens=200,
                temperature=0.1,
                candidate_count=1,
            )
            model_output = model.generate_content(contents=text, generation_config=generation_config)
            return model_output.candidates[0].content.parts[0].text
        except ResourceExhausted as e:
            sleep_time = 20
            print(f"Rate limit exceeded: {e}. Waiting for {sleep_time} seconds...")
            time.sleep(sleep_time)

In [2]:
from elasticsearch import Elasticsearch

es = Elasticsearch(os.getenv("ES_HOST"), basic_auth=(os.getenv("ES_USER"), os.getenv("ES_PASSWORD")))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

Connected to Elasticsearch


In [3]:
response = es.search(
    index="gumball_transcripts",
    query={"match_all": {}},
    size=10000
)

documents = response["hits"]["hits"]

print(f"Loaded {len(documents)} documents")

Loaded 256 documents


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def generate_image_prompt_parallel(texts, max_workers=len(api_keys)):
    prompts = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_text = {executor.submit(generate_image_prompt, text): text for text in texts}
        for future in as_completed(future_to_text):
            text = future_to_text[future]
            try:
                prompt = future.result()
                prompts.append(prompt)
            except Exception as exc:
                print(f"Text {text} generated an exception: {exc}")
                prompts.append(None)
    return prompts

# process every document. for every document's 10 \n separated texts, generate a prompt. the text is in the "text" field. put the new prompt in the "prompt" field in a different index
for doc in tqdm(documents, desc="Processing documents"):
    title = doc["_source"]["title"]
    text = doc["_source"]["text"]
    lines = text.split("\n")
    transcript = []
    chunks = ["\n".join(lines[i:i+10]).strip() for i in range(0, len(lines), 10) if "\n".join(lines[i:i+10]).strip()]
    
    prompts = generate_image_prompt_parallel(chunks)
    
    for chunk, prompt in zip(chunks, prompts):
        transcript.append({"text": chunk, "prompt": prompt})
    
    new_doc = {"title": title, "transcript": transcript}
    es.index(index="gumball_transcripts_with_prompts", document=new_doc)

Processing documents:   1%|          | 3/256 [00:13<18:30,  4.39s/it]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents:   3%|▎         | 8/256 [01:40<38:01,  9.20s/it]  

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents:   4%|▍         | 11/256 [02:38<52:42, 12.91s/it]  

...

Processing documents:  99%|█████████▉| 253/256 [1:07:37<00:42, 14.21s/it]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...
Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents: 100%|█████████▉| 255/256 [1:08:06<00:13, 13.50s/it]

Rate limit exceeded: 429 Resource has been exhausted (e.g. check quota).. Waiting for 20 seconds...


Processing documents: 100%|██████████| 256/256 [1:08:30<00:00, 16.06s/it]
